In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv("ps1small.csv")
df["lwage"] = np.log(df["wage"])

edu_levels = range(12, 18)
age_levels = range(26, 31)
cells = [(e, a) for e in edu_levels for a in age_levels]
print(cells)
print(len(cells))



[(12, 26), (12, 27), (12, 28), (12, 29), (12, 30), (13, 26), (13, 27), (13, 28), (13, 29), (13, 30), (14, 26), (14, 27), (14, 28), (14, 29), (14, 30), (15, 26), (15, 27), (15, 28), (15, 29), (15, 30), (16, 26), (16, 27), (16, 28), (16, 29), (16, 30), (17, 26), (17, 27), (17, 28), (17, 29), (17, 30)]
30


In [8]:
# One column per (education, age) cell; no intercept
D = np.column_stack([
    ((df["education"] == e) & (df["age"] == a)).astype(float)
    for e, a in cells
])
print(D.shape)
print(D[:5, :5])

y = df["lwage"].to_numpy()
print(y.shape)
print(y[:5])

(898, 30)
[[0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]
(898,)
[10.30895266  9.680344   10.68051622  9.79812704 10.1266311 ]


In [11]:
# OLS coefficients
alpha_hat = np.linalg.solve(D.T @ D, D.T @ y)

# Residual variance and standard errors
resid = y - D @ alpha_hat
n, k_u = D.shape
rss_u = resid @ resid
sigma2_hat = rss_u / (n - k_u)
vcov_alpha = sigma2_hat * np.linalg.inv(D.T @ D)
se_alpha = np.sqrt(np.diag(vcov_alpha))

results = pd.DataFrame({
    "education": [e for e, a in cells],
    "age": [a for e, a in cells],
    "n_cell": D.sum(axis=0).astype(int),
    "alpha_hat": alpha_hat,
    "std_error": se_alpha
})
print(results)
# export results to CSV
results.to_csv("ps2_results.csv", index=False)

    education  age  n_cell  alpha_hat  std_error
0          12   26      85   9.451320   0.090780
1          12   27      97   9.300426   0.084980
2          12   28     101   9.537856   0.083280
3          12   29      96   9.385826   0.085421
4          12   30      91   9.628323   0.087737
5          13   26      19   9.612120   0.192010
6          13   27      19   9.755901   0.192010
7          13   28      23   9.378158   0.174517
8          13   29      14   9.819982   0.223685
9          13   30      17   9.446488   0.202991
10         14   26      14   9.580658   0.223685
11         14   27      24   9.794423   0.170843
12         14   28      15   9.642237   0.216101
13         14   29      23   9.712199   0.174517
14         14   30      24   9.645440   0.170843
15         15   26      12   9.534555   0.241608
16         15   27      13   9.385619   0.232129
17         15   28      10   9.573155   0.264668
18         15   29      11   9.540361   0.252351
19         15   30  

In [14]:
import numpy as np
import pandas as pd

# Load data and create log wage
df = pd.read_csv("ps1small.csv")
df["lwage"] = np.log(df["wage"])

# Keep the ordering specified in the problem:
# alpha_12,26, ..., alpha_12,30, alpha_13,26, ..., alpha_17,30
education_levels = list(range(12, 18))
age_levels = list(range(26, 31))
cells = [
    (education, age)
    for education in education_levels
    for age in age_levels
]
cell_index = {cell: j for j, cell in enumerate(cells)}

# Define the four anchor cells
anchors = [(12, 26), (13, 26), (12, 27), (13, 27)]

# Construct R alpha = q.
# Each non-anchor cell's mean must equal the value implied by
# the bilinear model and the four anchor-cell means.
R_rows = []

for education, age in cells:
    if (education, age) in anchors:
        continue

    row = np.zeros(len(cells))

    # Coefficient on this cell mean
    row[cell_index[(education, age)]] += 1

    # Coefficients on the four anchor-cell means
    row[cell_index[(12, 26)]] -= (
        (1 - (education - 12)) * (1 - (age - 26))
    )
    row[cell_index[(13, 26)]] -= (
        (education - 12) * (1 - (age - 26))
    )
    row[cell_index[(12, 27)]] -= (
        (1 - (education - 12)) * (age - 26)
    )
    row[cell_index[(13, 27)]] -= (
        (education - 12) * (age - 26)
    )

    R_rows.append(row)

R = np.vstack(R_rows)
q = np.zeros((R.shape[0], 1))

print("R shape:", R.shape)             # (26, 30)
print("q shape:", q.shape)             # (26, 1)
print("Rank of R:", np.linalg.matrix_rank(R))  # 26

# Unrestricted model: 30 cell dummies, no intercept
D = np.column_stack([
    (
        (df["education"].to_numpy() == education)
        & (df["age"].to_numpy() == age)
    ).astype(float)
    for education, age in cells
])

y = df["lwage"].to_numpy()
alpha_hat = np.linalg.lstsq(D, y, rcond=None)[0]
resid_unrestricted = y - D @ alpha_hat
RSS_u = resid_unrestricted @ resid_unrestricted

# Restricted model:
# lwage = beta_0 + beta_1*education + beta_2*age
#         + beta_3*(education*age) + error
Z = np.column_stack([
    np.ones(len(df)),
    df["education"].to_numpy(),
    df["age"].to_numpy(),
    df["education"].to_numpy() * df["age"].to_numpy()
])

beta_hat = np.linalg.lstsq(Z, y, rcond=None)[0]
resid_restricted = y - Z @ beta_hat
RSS_r = resid_restricted @ resid_restricted

# F-test of the 26 restrictions
n = len(df)
k_unrestricted = D.shape[1]
p = R.shape[0]

F_stat = (
    ((RSS_r - RSS_u) / p)
    / (RSS_u / (n - k_unrestricted))
)

print("Restricted coefficients:", beta_hat)
print("RSS restricted:", RSS_r)
print("RSS unrestricted:", RSS_u)
print("Number of restrictions:", p)
print("Residual degrees of freedom:", n - k_unrestricted)
print("F statistic:", F_stat)

# 5% decision using the F(26, 868) critical value
F_critical_5pct = 1.5084
print("5% critical value:", F_critical_5pct)
print("Reject H0 at 5%:", F_stat > F_critical_5pct)

R shape: (26, 30)
q shape: (26, 1)
Rank of R: 26
Restricted coefficients: [ 6.10750540e+00  1.97807519e-01  6.48829069e-02 -2.51955614e-03]
RSS restricted: 624.7819133989759
RSS unrestricted: 608.0274467373475
Number of restrictions: 26
Residual degrees of freedom: 868
F statistic: 0.9199279217973295
5% critical value: 1.5084
Reject H0 at 5%: False


In [17]:
from pathlib import Path

# R must already be constructed as a NumPy array.
# Convert its entries to integers for cleaner LaTeX output.
row_strings = [
    " & ".join(str(int(value)) for value in row)
    for row in R
]

matrix_rows = r" \\" + "\n".join(row_strings)

latex_text = (
    "\\begin{center}\n"
    "\\resizebox{\\textwidth}{!}{$\n"
    "\\displaystyle R = \\begin{bmatrix}\n"
    + matrix_rows
    + "\n\\end{bmatrix},\\qquad "
    + "q=\\mathbf{0}_{26\\times 1}\n"
    + "$}\n"
    + "\\end{center}\n"
)

Path("R_matrix.tex").write_text(latex_text, encoding="utf-8")
print("Saved LaTeX matrix to R_matrix.tex")

Saved LaTeX matrix to R_matrix.tex


In [1]:
# number 3 - e

import numpy as np
import pandas as pd
from scipy.stats import f, norm

# Load the data
file_path = (
    r"C:\Users\chunghyunh\OneDrive - London Business School"
    r"\Desktop\LBS_Courses\2026AUT\Econometrics\wage.xlsx"
)
df = pd.read_excel(file_path)

# Keep valid observations and calculate log wage
df = df.loc[
    df["male"].isin([0, 1]) & (df["wage"] > 0),
    ["male", "wage"],
].copy()
df["log_wage"] = np.log(df["wage"])

# Group 1: females (male = 0)
# Group 2: males   (male = 1)
female = df.loc[df["male"] == 0, "log_wage"].to_numpy()
male = df.loc[df["male"] == 1, "log_wage"].to_numpy()

n_f, n_m = len(female), len(male)
s2_f = np.var(female, ddof=1)
s2_m = np.var(male, ddof=1)

# Test:
# H0: Var(log wage | female) = Var(log wage | male)
# H1: Var(log wage | female) > Var(log wage | male)

# ------------------------------------------------------------
# Standard normal-theory F test
# ------------------------------------------------------------
variance_ratio = s2_f / s2_m
f_critical = f.ppf(0.95, n_f - 1, n_m - 1)
p_value_f = f.sf(variance_ratio, n_f - 1, n_m - 1)

# ------------------------------------------------------------
# Heteroskedasticity/non-normality-robust asymptotic test
# ------------------------------------------------------------
T = np.sqrt(n_f * n_m / (n_f + n_m)) * np.log(variance_ratio)

# Pooled estimate of kappa from the question
numerator = (n_f + n_m) * (
    np.sum((female - female.mean()) ** 4)
    + np.sum((male - male.mean()) ** 4)
)
denominator = (
    np.sum((female - female.mean()) ** 2)
    + np.sum((male - male.mean()) ** 2)
) ** 2
kappa_hat = numerator / denominator

# Under H1, reject for large T / sqrt(kappa_hat - 1)
robust_stat = T / np.sqrt(kappa_hat - 1)
normal_critical = norm.ppf(0.95)
p_value_robust = norm.sf(robust_stat)

# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------
print(f"Number of female observations: {n_f}")
print(f"Number of male observations:   {n_m}")
print()
print(f"Female log-wage variance: {s2_f:.6f}")
print(f"Male log-wage variance:   {s2_m:.6f}")
print(f"Variance ratio (female / male): {variance_ratio:.6f}")
print()
print("Standard normal-theory F test")
print(f"  F statistic:       {variance_ratio:.6f}")
print(f"  5% critical value: {f_critical:.6f}")
print(f"  One-sided p-value: {p_value_f:.6g}")
print(f"  Reject at 5%:      {variance_ratio > f_critical}")
print()
print("Robust asymptotic test")
print(f"  T statistic:              {T:.6f}")
print(f"  Estimated kappa:           {kappa_hat:.6f}")
print(f"  Robust statistic:          {robust_stat:.6f}")
print(f"  5% critical value:         {normal_critical:.6f}")
print(f"  One-sided p-value:         {p_value_robust:.6g}")
print(f"  Reject at 5%:              {robust_stat > normal_critical}")


Number of female observations: 1569
Number of male observations:   1727

Female log-wage variance: 0.397877
Male log-wage variance:   0.366413
Variance ratio (female / male): 1.085871

Standard normal-theory F test
  F statistic:       1.085871
  5% critical value: 1.084464
  One-sided p-value: 0.0473466
  Reject at 5%:      True

Robust asymptotic test
  T statistic:              2.362096
  Estimated kappa:           7.378121
  Robust statistic:          0.935300
  5% critical value:         1.644854
  One-sided p-value:         0.174817
  Reject at 5%:              False
